# Phase 0: Manhattan Distance Verification / supervised rollout

**Goal**: Verify that MLP and CNN can learn Manhattan distance from the
4-channel grid encoding before introducing movement.

If the model cannot learn Manhattan distance, all downstream experiments
will have unexplained bias (see Environment Dynamics doc, Section 2.3).

**Setup**: Supervised regression. Place 1 apple and 2 agents randomly on a
6×6 grid. Predict [dist(self, apple), dist(other, apple)]. MSE loss.

**Success criterion**: <1% relative error on held-out test set.

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================

MODEL_TYPE = "cnn"          # "mlp" | "cnn"

# Environment (matching Phase 1 parameters)
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
NUM_APPLES = 1

# Training
LR = 0.001
LR_SCHEDULE = 'halve'        # 'halve' | 'none'
LR_HALVE_INTERVAL = 10000
LR_MIN = 1e-6
TRAIN_STEPS = 100000
EVAL_FREQ = 500
BATCH_SIZE = 64

# Model architecture
MLP_LAYERS = [64, 64]
CNN_LAYERS = [(16, 3), (16, 3)]
CNN_MLP_HEAD_LAYERS = [2]

# Dataset
N_TRAIN = 50000
N_TEST = 5000

# Output
SEED = 42
OUTPUT_PATH = "outputs"

TAG = "default"

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import gc
import os
import ast
import json
import time

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import numpy as np
import pandas as pd

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf

from helpers_forcedstay import (
    StateForcedStay,
    init_apple_positions,
    init_agent_positions,
    encode_state,
    create_model,
    manhattan_distance,
    get_memory_mb,
    get_weight_stats,
    update_lr,
)

# Set seeds
rng = np.random.default_rng(SEED)
tf.random.set_seed(SEED)

# Papermill safety
if isinstance(MLP_LAYERS, str):
    MLP_LAYERS = ast.literal_eval(MLP_LAYERS)
if isinstance(CNN_LAYERS, str):
    CNN_LAYERS = ast.literal_eval(CNN_LAYERS)
if isinstance(CNN_MLP_HEAD_LAYERS, str):
    CNN_MLP_HEAD_LAYERS = ast.literal_eval(CNN_MLP_HEAD_LAYERS)

INPUT_SHAPE = (HEIGHT, WIDTH, 4)
OUTPUT_DIM = 2  # [dist(self, apple), dist(other, apple)]

print(f"Model: {MODEL_TYPE}")
print(f"Grid: {HEIGHT}x{WIDTH}, Agents: {NUM_AGENTS}, Apples: {NUM_APPLES}")
print(f"Input shape: {INPUT_SHAPE}, Output dim: {OUTPUT_DIM}")
print(f"Train samples: {N_TRAIN}, Test samples: {N_TEST}")
print(f"Verified Config - MLP: {MLP_LAYERS} ({type(MLP_LAYERS)}), CNN: {CNN_LAYERS} ({type(CNN_LAYERS)})")
print(f"LR: {LR}, Schedule: {LR_SCHEDULE}, Halve every: {LR_HALVE_INTERVAL}, Min: {LR_MIN}")

start_time = time.time()

print(f"run started at {time.ctime(start_time)}")

Model: cnn
Grid: 6x6, Agents: 2, Apples: 1
Input shape: (6, 6, 4), Output dim: 2
Train samples: 50000, Test samples: 5000
Verified Config - MLP: [64, 64] (<class 'list'>), CNN: [(16, 3), (16, 3)] (<class 'list'>)
LR: 0.001, Schedule: halve, Halve every: 10000, Min: 1e-06
run started at Tue Feb 10 09:55:40 2026


In [3]:
# =============================================================================
# DATASET GENERATION
# =============================================================================

def generate_dataset(n_samples, rng):
    """
    Generate (encoding, target) pairs for Manhattan distance prediction.

    For each sample:
    - Place 1 apple randomly
    - Place 2 agents randomly (distinct cells, may overlap with apple)
    - Randomly assign actor (0 or 1)
    - Encode from agent 0's perspective
    - Target: [dist(agent_0, apple), dist(agent_1, apple)]
    """
    encodings = []
    targets = []

    for _ in range(n_samples):
        # Random apple position
        apple_pos = (int(rng.integers(0, HEIGHT)), int(rng.integers(0, WIDTH)))
        apple_set = {apple_pos}

        # Random agent positions (distinct cells)
        agent_pos = init_agent_positions(NUM_AGENTS, HEIGHT, WIDTH, rng, exclude=None)

        # Random actor
        actor = int(rng.integers(0, NUM_AGENTS))

        state = StateForcedStay(
            agent_positions=agent_pos,
            apple_positions=apple_set,
            actor=actor,
            H=HEIGHT,
            W=WIDTH,
        )

        # Encode from agent 0's perspective
        enc = encode_state(state, self_idx=0)
        encodings.append(enc)

        # Targets: Manhattan distances
        d_self = manhattan_distance(agent_pos[0], apple_pos)
        d_other = manhattan_distance(agent_pos[1], apple_pos)
        targets.append([d_self, d_other])

    return np.array(encodings, dtype=np.float32), np.array(targets, dtype=np.float32)


print("Generating training set...")
X_train, y_train = generate_dataset(N_TRAIN, rng)

print("Generating test set...")
test_rng = np.random.default_rng(SEED + 1000)
X_test, y_test = generate_dataset(N_TEST, test_rng)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

# Sanity check: distance ranges
print(f"Target stats (train):")
print(f"  dist(self,apple):  min={y_train[:,0].min():.0f}, max={y_train[:,0].max():.0f}, mean={y_train[:,0].mean():.2f}")
print(f"  dist(other,apple): min={y_train[:,1].min():.0f}, max={y_train[:,1].max():.0f}, mean={y_train[:,1].mean():.2f}")

Generating training set...
Generating test set...
X_train: (50000, 6, 6, 4), y_train: (50000, 2)
X_test:  (5000, 6, 6, 4),  y_test:  (5000, 2)
Target stats (train):
  dist(self,apple):  min=0, max=10, mean=3.90
  dist(other,apple): min=0, max=10, mean=3.89


In [4]:
# =============================================================================
# CREATE MODEL
# =============================================================================

model = create_model(INPUT_SHAPE, MODEL_TYPE, MLP_LAYERS, CNN_LAYERS, CNN_MLP_HEAD_LAYERS, LR,
                     output_dim=OUTPUT_DIM)
print(f"Created {MODEL_TYPE.upper()} model")
model.summary()

Created CNN model


/u/taddmao/code/orchard-action-market/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-10 09:55:44.038091: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 6, 6, 16)       │           592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 6, 6, 16)       │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         1,154 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,072 (15.91 KB)

 Trainable params: 4,072 (15.91 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# Scale information
print(f"\n--- Test Set Target Statistics ---")
print(f"  dist(self,apple):  mean={y_test[:,0].mean():.4f}, std={y_test[:,0].std():.4f}, min={y_test[:,0].min():.0f}, max={y_test[:,0].max():.0f}")
print(f"  dist(other,apple): mean={y_test[:,1].mean():.4f}, std={y_test[:,1].std():.4f}, min={y_test[:,1].min():.0f}, max={y_test[:,1].max():.0f}")
print(f"  combined:          mean={y_test.mean():.4f}, std={y_test.std():.4f}")
print(f"\n  For <1% relative error, need MAE < {y_test.mean() * 0.01:.4f}")


--- Test Set Target Statistics ---
  dist(self,apple):  mean=3.8888, std=2.0407, min=0, max=10
  dist(other,apple): mean=3.8182, std=2.0411, min=0, max=10
  combined:          mean=3.8535, std=2.0412

  For <1% relative error, need MAE < 0.0385


In [6]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

os.makedirs(OUTPUT_PATH, exist_ok=True)
csv_path = f"{OUTPUT_PATH}/phase0_{TAG}_grid{HEIGHT}.csv"

# Metadata
metadata = {
    "phase": 0,
    "task": "manhattan_distance",
    "MODEL_TYPE": MODEL_TYPE,
    "HEIGHT": HEIGHT,
    "WIDTH": WIDTH,
    "NUM_AGENTS": NUM_AGENTS,
    "NUM_APPLES": NUM_APPLES,
    "LR": LR,
    "LR_SCHEDULE": LR_SCHEDULE,
    "LR_HALVE_INTERVAL": LR_HALVE_INTERVAL,
    "LR_MIN": LR_MIN,
    "TRAIN_STEPS": TRAIN_STEPS,
    "EVAL_FREQ": EVAL_FREQ,
    "BATCH_SIZE": BATCH_SIZE,
    "MLP_LAYERS": MLP_LAYERS,
    "CNN_LAYERS": CNN_LAYERS,
    "N_TRAIN": N_TRAIN,
    "N_TEST": N_TEST,
    "SEED": SEED,
    "OUTPUT_PATH": OUTPUT_PATH,
}
with open(csv_path.replace('.csv', '_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

first_write = True
prev_weights = []

print(f"Training: {MODEL_TYPE}")
print(f"Steps: {TRAIN_STEPS}, Eval every: {EVAL_FREQ}")
print(f"Logging to: {csv_path}")

for step in range(TRAIN_STEPS):
    current_lr = update_lr(step, [model], LR,
                           schedule=LR_SCHEDULE,
                           interval=LR_HALVE_INTERVAL,
                           min_lr=LR_MIN)

    # Sample mini-batch
    idx = rng.choice(N_TRAIN, size=BATCH_SIZE, replace=False)
    X_batch = tf.constant(X_train[idx])
    y_batch = tf.constant(y_train[idx])

    loss = model.train_on_batch(X_batch, y_batch)

    # Evaluate
    if step % EVAL_FREQ == 0:
        # Predict on full test set
        preds = model(tf.constant(X_test), training=False).numpy()

        # Per-output metrics
        errors = preds - y_test
        abs_errors = np.abs(errors)

        mae_self = float(np.mean(abs_errors[:, 0]))
        mae_other = float(np.mean(abs_errors[:, 1]))
        mae_total = float(np.mean(abs_errors))

        bias_self = float(np.mean(errors[:, 0]))
        bias_other = float(np.mean(errors[:, 1]))

        max_err_self = float(np.max(abs_errors[:, 0]))
        max_err_other = float(np.max(abs_errors[:, 1]))

        # Percent error (relative to mean true distance)
        mean_true_self = float(np.mean(y_test[:, 0]))
        mean_true_other = float(np.mean(y_test[:, 1]))
        pct_err_self = (mae_self / mean_true_self * 100) if mean_true_self > 1e-10 else float('nan')
        pct_err_other = (mae_other / mean_true_other * 100) if mean_true_other > 1e-10 else float('nan')
        pct_err_total = ((mae_self + mae_other) / (mean_true_self + mean_true_other) * 100)

        weight_stats, prev_weights = get_weight_stats(model, prev_weights)

        row = {
            'step': step,
            'train_loss': float(loss),
            'mae_self': mae_self,
            'mae_other': mae_other,
            'mae_total': mae_total,
            'bias_self': bias_self,
            'bias_other': bias_other,
            'max_err_self': max_err_self,
            'max_err_other': max_err_other,
            'pct_err_self': pct_err_self,
            'pct_err_other': pct_err_other,
            'pct_err_total': pct_err_total,
            'current_lr': current_lr,
            'memory_mb': get_memory_mb(),
            **weight_stats,
        }

        pd.DataFrame([row]).to_csv(
            csv_path,
            mode='w' if first_write else 'a',
            header=first_write,
            index=False,
        )
        first_write = False

        if step % (EVAL_FREQ * 10) == 0:
            print(f"Step {step}: loss={loss:.4f}, "
                  f"mae_self={mae_self:.3f} ({pct_err_self:.2f}%), "
                  f"mae_other={mae_other:.3f} ({pct_err_other:.2f}%)")

print("Training complete!")
print(f"Results saved to: {csv_path}")

Training: cnn
Steps: 100000, Eval every: 500
Logging to: outputs/phase0_default_grid6.csv
Step 0: Learning Rate changed to 1.00e-03
Step 0: loss=21.3432, mae_self=3.886 (99.93%), mae_other=3.818 (100.00%)
Step 1: Learning Rate changed to 1.00e-03
Step 2: Learning Rate changed to 1.00e-03
Step 3: Learning Rate changed to 1.00e-03
Step 4: Learning Rate changed to 1.00e-03
Step 5: Learning Rate changed to 1.00e-03
Step 6: Learning Rate changed to 1.00e-03
Step 7: Learning Rate changed to 1.00e-03
Step 8: Learning Rate changed to 1.00e-03
Step 9: Learning Rate changed to 1.00e-03
Step 10: Learning Rate changed to 1.00e-03
Step 11: Learning Rate changed to 1.00e-03
Step 12: Learning Rate changed to 1.00e-03
Step 13: Learning Rate changed to 1.00e-03
Step 14: Learning Rate changed to 1.00e-03
Step 15: Learning Rate changed to 1.00e-03
Step 16: Learning Rate changed to 1.00e-03
Step 17: Learning Rate changed to 1.00e-03
Step 18: Learning Rate changed to 1.00e-03
Step 19: Learning Rate changed

KeyboardInterrupt: 

In [ ]:
# =============================================================================
# FINAL EVALUATION
# =============================================================================

preds = model(tf.constant(X_test), training=False).numpy()
errors = preds - y_test
abs_errors = np.abs(errors)

print("=" * 60)
print("FINAL TEST SET RESULTS")
print("=" * 60)

for col_idx, name in enumerate(["dist(self, apple)", "dist(other, apple)"]):
    mae = np.mean(abs_errors[:, col_idx])
    bias = np.mean(errors[:, col_idx])
    max_err = np.max(abs_errors[:, col_idx])
    mean_true = np.mean(y_test[:, col_idx])
    pct = mae / mean_true * 100 if mean_true > 1e-10 else float('nan')

    print(f"{name}:")
    print(f"  MAE:        {mae:.4f}")
    print(f"  Bias:       {bias:.4f}")
    print(f"  Max Error:  {max_err:.4f}")
    print(f"  Mean True:  {mean_true:.4f}")
    print(f"  % Error:    {pct:.2f}%")

overall_pct = np.mean(abs_errors) / np.mean(y_test) * 100
print(f"Overall % Error: {overall_pct:.2f}%")

if overall_pct < 1.0:
    print("*** PASS: <1% relative error ***")
else:
    print("*** FAIL: >=1% relative error -- investigate before proceeding ***")

In [ ]:
# =============================================================================
# TIMING
# =============================================================================

end_time = time.time()
total_duration = end_time - start_time

metadata["total_time_seconds"] = total_duration
metadata["total_time_readable"] = time.strftime("%H:%M:%S", time.gmtime(total_duration))

with open(csv_path.replace('.csv', '_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"Total time: {metadata['total_time_readable']}")